In [1]:
# ruff: noqa: F401
import os
import subprocess
from pathlib import Path

import board
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.models import FallbackModel
from google.adk.runners import InMemoryRunner
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from google.genai import types
from mcp import StdioServerParameters
from quiet import silence

silence()  # mute the messages from library logging

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")

load_dotenv(override=True)

MODEL1 = "gemini-3.1-flash-lite"
MODEL2 = "gemini-3-flash-preview"


/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Create agent


In [2]:
agent = LlmAgent(
    model=FallbackModel(models=[MODEL1, MODEL2]),
    name="assistant",
    instruction="You are a concise, friendly assistant. Reply in short, one sentence",
)


### Run the agent


In [3]:
result = await InMemoryRunner(agent=agent).run_debug("Say hello in French", verbose=True)

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.
Model gemini-3.1-flash-lite failed with status 503; falling back to the next model.


assistant > Bonjour, I hope you're having a wonderful day!


In [4]:
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into Bengali and write the results in bengali.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into Bengali and write the results in bengali.txt',
  'status': 'pending',
  'result': ''}]

### Add tools


In [5]:
def show_todos() -> list[dict]:
    """List all todos on board. A goal has a parent id as None. A step has parent id set to its goal id"""
    return board.list_todos()


def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into ordered list of stpes. Pass the goal-id and a short description to each step"""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}


def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo or a a goal as done and provide a short summary"""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "result": "done"}


def send_push_notification(message: str) -> str:
    """Send a short push notification to the user's phone to report a result or that a job is done."""
    payload = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": message,
        "sound": "climb",
    }
    response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
    return f"Push notification sent (status {response.status_code})."

In [6]:
board_agent = LlmAgent(
    model=MODEL1, name="board_agent", instruction="Help manage a shared todo board", tools=[show_todos, complete_task]
)

result = await InMemoryRunner(agent=board_agent).run_debug("What is on board right now and what is the status of each?")

Node execution failed with exception
Traceback (most recent call last):
  File "/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-packages/google/adk/workflow/_node_runner.py", line 137, in run
    await self._execute_node(ctx, node_input)
  File "/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-packages/google/adk/workflow/_node_runner.py", line 285, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-packages/google/adk/workflow/_node_runner.py", line 299, in _run_node_loop
    async for event in agen:
      self._track_event_in_context(event, ctx)
      await self._enqueue_event(event, ctx)
  File "/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-packages/google/adk/workflow/_base_node.py", line 175, in run
    async for item in agen:
    ...<12 lines>...
        yield Event(output=validated)
  File "/Users/AnirbanGuha/venvs/agent-mcp-course/lib/python3.14/site-pac

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### Using MCP


In [ ]:
workspace = Path("taskworker/workspace").resolve()

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),
        ),
        timeout=60,
    ),
    errlog=subprocess.DEVNULL,
)


In [ ]:
file_agent = LlmAgent(
    model=FallbackModel(models=[MODEL1, MODEL2]),
    name="file_agent",
    instruction="You can read and write files in your workspace. Use tools to accomplish the tasks",
    tools=[filesystem],
)

In [ ]:
result = await InMemoryRunner(agent=file_agent).run_debug(
    "Read notes.txt and summarize the content in one sentence", verbose=True
)

file_agent > [Calling tool: list_allowed_directories({})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Allowed directories:\n/Users/AnirbanGuha/Library/CloudStorage...]
file_agent > [Calling tool: list_directory({'path': '.'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': '[FILE] notes.txt'}], 'structuredContent': {'content': '[FILE]...]
file_agent > [Calling tool: read_text_file({'path': 'notes.txt'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\n\nToday we are building a small languag...]
file_agent > The team is collaborating to build a language tutor by having individual helpers carefully complete and mark off specific tasks from a shared board.


### Run an agent loop


In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.
Take the pending goal and see it through. Begin by laying out a short plan:
the handful of concrete steps the work itself breaks down into, added to the
board under the goal. Then carry them out with your file tools, marking each
step done as you finish it. Once the steps are all done, close the goal. Your
files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=FallbackModel(models=[MODEL1, MODEL2]),
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem, send_push_notification],
)

board.reset_board()
goal_id = board.add_goal(
    "Read notes.txt, translate the content into Bengali and write the translated text into bengali.txt"
)

# Drive the loop ourselves with run_async instead of run_debug, so we can watch it:
# print each tool call, and re-draw the board whenever a step is planned or ticked off.
runner = InMemoryRunner(agent=worker)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ag")

async for event in runner.run_async(
    user_id="ag",
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board."),
):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
    if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
        board.show_board()
    if event.is_final_response():
        print(event.content.parts[0].text)

tool: show_todos({})
tool: list_allowed_directories({})
tool: list_directory({'path': '/Users/AnirbanGuha/Library/CloudStorage/GoogleDrive-guhaa1@gmail.com/My Drive/GitHub/The Complete Agent and MCP Course/Exercises/agent_frameworks/01_google_adk_a2a/taskworker/workspace'})
tool: plan_steps({'goal_id': 1, 'steps': ['Read notes.txt', 'Translate the content to Bengali', 'Write the translated text to bengali.txt']})


Goal #1: Read notes.txt, translate the content into Bengali and write the translated text into bengali.txt
  Step #2: Read notes.txt
  Step #3: Translate the content to Bengali
  Step #4: Write the translated text to bengali.txt

tool: read_text_file({'path': '/Users/AnirbanGuha/Library/CloudStorage/GoogleDrive-guhaa1@gmail.com/My Drive/GitHub/The Complete Agent and MCP Course/Exercises/agent_frameworks/01_google_adk_a2a/taskworker/workspace/notes.txt'})
tool: complete_task({'task_id': 2, 'result': 'Read notes.txt successfully.'})


Goal #1: Read notes.txt, translate the content into Bengali and write the translated text into bengali.txt
  Step #2: Read notes.txt  Read notes.txt successfully.
  Step #3: Translate the content to Bengali
  Step #4: Write the translated text to bengali.txt

tool: complete_task({'task_id': 3, 'result': 'Translated the content:\nআমাদের দলে স্বাগতম।\n\nআজ আমরা একটি ছোট ভাষা প্রশিক্ষক তৈরি করছি, একে একে। প্রতিটি সহায়ক শেয়ার করা বোর্ড থেকে একটি একক কাজ বেছে নেয়, কাজটিকে সাবধানে সম্পন্ন করে এবং কাজটি সম্পন্ন হয়েছে বলে চিহ্নিত করে। আপনার কাজটি মনোযোগ দিয়ে পড়ুন, সময় নিন এবং আপনার সেরাটা করুন।'})


...

tool: write_file({'path': '/Users/AnirbanGuha/Library/CloudStorage/GoogleDrive-guhaa1@gmail.com/My Drive/GitHub/The Complete Agent and MCP Course/Exercises/agent_frameworks/01_google_adk_a2a/taskworker/workspace/bengali.txt', 'content': 'আমাদের দলে স্বাগতম।\n\nআজ আমরা একটি ছোট ভাষা প্রশিক্ষক তৈরি করছি, একে একে। প্রতিটি সহায়ক শেয়ার করা বোর্ড থেকে একটি একক কাজ বেছে নেয়, কাজটিকে সাবধানে সম্পন্ন করে এবং কাজটি সম্পন্ন হয়েছে বলে চিহ্নিত করে। আপনার কাজটি মনোযোগ দিয়ে পড়ুন, সময় নিন এবং আপনার সেরাটা করুন।'})
tool: complete_task({'result': 'Successfully translated and wrote the file to bengali.txt.', 'task_id': 4})


Goal #1: Read notes.txt, translate the content into Bengali and write the translated text into bengali.txt
  Step #2: Read notes.txt  Read notes.txt successfully.
  Step #3: Translate the content to Bengali  Translated the content:
আমাদের দলে স্বাগতম।

আজ আমরা একটি ছোট ভাষা প্রশিক্ষক তৈরি করছি, একে একে। প্রতিটি সহায়ক শেয়ার করা বোর্ড থেকে একটি একক কাজ বেছে নেয়, কাজটিকে সাবধানে সম্পন্ন করে এবং কাজটি সম্পন্ন হয়েছে বলে চিহ্নিত করে। আপনার কাজটি মনোযোগ দিয়ে পড়ুন, সময় নিন এবং আপনার সেরাটা করুন।
  Step #4: Write the translated text to bengali.txt  Successfully translated and wrote the file to bengali.txt.

tool: complete_task({'result': 'All steps completed for the goal. Read notes.txt, translated to Bengali, and saved as bengali.txt.', 'task_id': 1})


Goal #1: Read notes.txt, translate the content into Bengali and write the translated text into bengali.txt  All steps completed for the goal. Read notes.txt, translated to Bengali, and saved as bengali.txt.
  Step #2: Read notes.txt  Read notes.txt successfully.
  Step #3: Translate the content to Bengali  Translated the content:
আমাদের দলে স্বাগতম।

আজ আমরা একটি ছোট ভাষা প্রশিক্ষক তৈরি করছি, একে একে। প্রতিটি সহায়ক শেয়ার করা বোর্ড থেকে একটি একক কাজ বেছে নেয়, কাজটিকে সাবধানে সম্পন্ন করে এবং কাজটি সম্পন্ন হয়েছে বলে চিহ্নিত করে। আপনার কাজটি মনোযোগ দিয়ে পড়ুন, সময় নিন এবং আপনার সেরাটা করুন।
  Step #4: Write the translated text to bengali.txt  Successfully translated and wrote the file to bengali.txt.

The pending goal is complete. I read the content of `notes.txt`, translated it into Bengali, and saved the translation to `bengali.txt`. All associated steps have been marked as done.
